# Retrieve metadata for a validation process

This tutorial shows how pysdmx can be used to retrieve the structures required to validate data.

## Step 1: Connect to the service that contains the required structures

The client can connect to any compatible SDMX-RESTv2 service.

In [ ]:
from pathlib import Path

from pysdmx.api.fmr import RegistryClient
from pysdmx.io import read_sdmx

client = RegistryClient("https://registry.sdmx.org/sdmx/v2")

## Step 2: Retrieve the required schema

In [ ]:
schema = client.get_schema("dataflow", "ESTAT", "BPM6_BOP_M", "2.6.0")
print(schema)

## Step 3: Inspect the schema

Notice the list of components, their data type, enumerations, and facets.

In [ ]:
print([component.id for component in schema.components])

frequency = schema.components["FREQ"]
print(f"Data type: {frequency.dtype}")
print(f"Enumeration: {[code.id for code in frequency.enumeration]}")
print(f"Facets: {frequency.facets}")

Notice the list of structures used to generate the schema (`artefacts`).

In [ ]:
for artefact in schema.artefacts:
    print(artefact)

Notice the attribute relationships.

In [ ]:
for attribute, relationship in schema.attribute_relationships.items():
    print(f"{attribute}: {relationship.name}")

## Step 4: Load a data file and inspect its data

Note: The CSV file was generated automatically, i.e. there is no guarantee the data are meaningful.

In [ ]:
file_path = Path("../samples/bpm6_bop_m.csv")
if not file_path.exists():
    file_path = Path("docs/tutorials/samples/bpm6_bop_m.csv")
submission = read_sdmx(file_path)
display(submission.data[0])

As you can see, the data are represented as Pandas DataFrames.

## Write your own code to validate a data message using the schema

Note: pysdmx will soon offer a `validate()` function!

In [ ]:
for ds in submission.data:
    df = ds.data

    # For example, check that values belong to the relevant enumerations.
    for column in df.columns:
        component = schema.components[column]
        if component and component.enumeration:
            allowed_values = {code.id for code in component.enumeration}
            values = {value for value in df[column].dropna() if value != ""}
            unexpected = values - allowed_values
            if unexpected:
                print(f"Found invalid values for {component.id}: {unexpected}")